# Hyperparameter Optimization 

This notebook performs **hyperparameter optimization** for the `PeakPredictorLSTM` model using **Optuna**.

The objective is to identify the best configuration of model, training, and data parameters that maximizes **PR-AUC (Precision–Recall AUC)** on a validation dataset generated from synthetic time series.

---

#### Objective

- Optimize model performance for **peak detection in time series**
- Maximize **PR-AUC** on the validation set
- Evaluate robustness across different window sizes and model capacities

---

#### Dataset

- **Type:** Synthetic time series generated using a parametric *p-model*
- **Training set:** Multiple generated time series
- **Validation set:** Independent generated time series
- **Class imbalance:** Addressed using weighted binary cross-entropy

---

#### Optimization Setup

- **Optimizer:** Adam
- **Loss function:** `BCEWithLogitsLoss` with dynamic positive class weighting
- **Early stopping:** Enabled based on validation loss
- **Device:** CPU / CUDA / Apple MPS (automatically selected)

---

#### Evaluation Metric

- **Primary metric:** PR-AUC (Average Precision Score)
- Computed on the validation set after training

---

#### Author

**Carlos Eduardo Falandes**  
M.Sc. Student in Applied Computing  
National Institute for Space Research (INPE)  
📧 carlos.falandes@inpe.br


In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path().resolve().parents[0]
sys.path.append(str(ROOT_DIR))


In [2]:
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import numpy as np
from sklearn.metrics import average_precision_score

import optuna

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

from src.models.model import PeakPredictorLSTM
from src.data.pmodel_generation import (
    generate_multiple_series,
    create_sequences_from_multiple_series
)


In [3]:
def prepare_dataloader_from_series(series_list, window_size, lookahead, batch_size):
    X, y = create_sequences_from_multiple_series(
        series_list, window_size, lookahead
    )

    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )

    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [4]:
def compute_pos_weight(dataloader, device):
    total_pos, total_neg = 0, 0
    for _, y in dataloader:
        total_pos += (y == 1).sum().item()
        total_neg += (y == 0).sum().item()
    return torch.tensor([total_neg / total_pos], device=device)

In [5]:
def train_model(
    model,
    criterion,
    optimizer,
    train_loader,
    val_loader,
    epochs,
    device,
    patience=10
):
    """
    Trains a PyTorch model with early stopping based on validation loss.

    Args:
        model: PyTorch model to train
        criterion: Loss function
        optimizer: Optimizer
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        epochs (int): Maximum number of epochs
        device: torch.device ("cpu" or "cuda")
        patience (int): Early stopping patience

    Returns:
        dict: Training history with train and validation losses
    """

    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        # -------- Training --------
        model.train()
        train_loss = 0.0

        for X_batch, y_batch in tqdm(
            train_loader,
            desc=f"Epoch {epoch}/{epochs} - Training",
            disable=True
        ):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch).squeeze(-1)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # -------- Validation --------
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val = X_val.to(device)
                y_val = y_val.to(device)

                outputs = model(X_val).squeeze(-1)
                loss = criterion(outputs, y_val)
                val_loss += loss.item()

        val_loss /= len(val_loader)

        # -------- Early Stopping --------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping triggered after {epoch} epochs")
            break

    print("\nTraining finished")
    return history


In [ ]:
def objective(trial):
    """
    Optuna objective function for hyperparameter optimization.
    Returns PR-AUC on the validation set.
    """

    # -------- Load configuration --------
    with open("../config/config_exo_endo.json", "r") as f:
        config = json.load(f)

    # -------- Device selection --------
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    # -------- Hyperparameter search space --------
    # config["training"]["learning_rate"] = trial.suggest_float(
    #     "learning_rate", 1e-4, 5e-3, log=True
    # )
    # config["training"]["batch_size"] = trial.suggest_categorical(
    #     "batch_size", [64, 128, 256]
    # )

    # config["model"]["hidden_dim"] = trial.suggest_categorical(
    #     "hidden_dim", [64, 128, 256]
    # )
    # config["model"]["num_layers"] = trial.suggest_int(
    #     "num_layers", 1, 3
    # )
    # config["model"]["dropout"] = trial.suggest_float(
    #     "dropout", 0.1, 0.4
    # )

    # config["data"]["window_size"] = trial.suggest_categorical(
    #     "window_size", [256, 384, 512]
    # )

    config["training"]["batch_size"] = trial.suggest_categorical(
        "batch_size", [64, 128, 256]
    )

    config["model"]["hidden_dim"] = trial.suggest_categorical(
        "hidden_dim", [64, 128, 256]
    )
    config["model"]["num_layers"] = trial.suggest_categorical(
        "num_layers", [1, 2, 3]
    )

    config["model"]["projection_dim"] = trial.suggest_categorical(
        "projection_dim", [16, 32, 64]
    )

    config["data"]["window_size"] = trial.suggest_categorical(
        "window_size", [256, 384, 512]
    )


    # -------- Data generation --------
    train_series = generate_multiple_series(
        length=config["p_model"]["series_length"],
        p_value=config["p_model"]["p_value"],
        num_series=1000,
        peak_percentile=config["data"]["peak_percentile"],
    )

    val_series = generate_multiple_series(
        length=config["p_model"]["series_length"],
        p_value=config["p_model"]["p_value"],
        num_series=300,
        peak_percentile=config["data"]["peak_percentile"],
    )

    # -------- Dataloaders --------
    train_loader = prepare_dataloader_from_series(
        series_list=train_series,
        window_size=config["data"]["window_size"],
        lookahead=config["data"]["lookahead"],
        batch_size=config["training"]["batch_size"],
    )

    val_loader = prepare_dataloader_from_series(
        series_list=val_series,
        window_size=config["data"]["window_size"],
        lookahead=config["data"]["lookahead"],
        batch_size=config["training"]["batch_size"],
    )

    # -------- Model --------
    model = PeakPredictorLSTM(config["model"]).to(device)

    # -------- Loss function --------
    raw_pos_weight = compute_pos_weight(train_loader, device)
    pos_weight = torch.clamp(raw_pos_weight, max=7.0)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # -------- Optimizer --------
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["training"]["learning_rate"]
    )

    # -------- Training --------
    train_model(
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=60,
        device=device,
        patience=5,
    )

    # -------- Evaluation (PR-AUC) --------
    model.eval()
    all_probs, all_targets = [], []

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            logits = model(X).squeeze(-1)
            probs = torch.sigmoid(logits).cpu().numpy()

            all_probs.extend(probs)
            all_targets.extend(y.numpy())

    pr_auc = average_precision_score(all_targets, all_probs)

    return pr_auc


In [ ]:
with open("../config/config_endo_exo.json", "r") as f:
    config = json.load(f)

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

study.optimize(objective, n_trials=100)
print(f"\nP-model series length: {config['p_model']['series_length']}\n")

print("Best hyperparameters:")
print(study.best_params)



[I 2026-01-19 17:43:37,389] A new study created in memory with name: no-name-e17183fc-35ca-4602-9e77-3c601a859304



Early stopping triggered after 27 epochs

Training finished


[I 2026-01-19 17:47:52,301] Trial 0 finished with value: 0.2930703769220413 and parameters: {'batch_size': 128, 'hidden_dim': 64, 'num_layers': 1}. Best is trial 0 with value: 0.2930703769220413.



Early stopping triggered after 14 epochs

Training finished


[I 2026-01-19 17:53:04,019] Trial 1 finished with value: 0.3275700755598638 and parameters: {'batch_size': 64, 'hidden_dim': 128, 'num_layers': 1}. Best is trial 0 with value: 0.2930703769220413.


...


Early stopping triggered after 17 epochs

Training finished


[I 2026-01-19 22:42:29,409] Trial 23 finished with value: 0.3000643531365239 and parameters: {'batch_size': 64, 'hidden_dim': 64, 'num_layers': 1}. Best is trial 0 with value: 0.2930703769220413.



Early stopping triggered after 25 epochs

Training finished


[I 2026-01-19 22:50:08,000] Trial 24 finished with value: 0.32610730905816576 and parameters: {'batch_size': 64, 'hidden_dim': 64, 'num_layers': 1}. Best is trial 0 with value: 0.2930703769220413.



P-model series length: 1024

Best hyperparameters:
{'batch_size': 128, 'hidden_dim': 64, 'num_layers': 1}
